# Step 4: Supervised Baseline U-Net + Step 5: Limited-Label Settings

**Objectives**:
1. Train a U-Net baseline with random initialization on 100% labels
2. Evaluate: per-class Dice, HD95, mean Dice on test set
3. Train on limited labels (10%, 25%, 50%) and measure performance drop
4. Generate label-efficiency curve

**Architecture**: MONAI-style 2D U-Net [32, 64, 128, 256]  
**Loss**: Dice + Cross-Entropy  
**Optimizer**: AdamW, LR=1e-4, WD=1e-5  
**This establishes the baseline that SSL pretraining must improve upon.**

In [ ]:
import sys
import os
import json
import numpy as np
import torch
import matplotlib.pyplot as plt
import pandas as pd
from pathlib import Path
from torch.utils.data import DataLoader

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.dataset import ACDCSegDataset, get_train_transforms, get_val_transforms
from src.segmentation_model import SegmentationUNet
from src.losses import DiceCELoss
from src.metrics import (
    compute_metrics_batch, compute_patient_level_metrics, format_metrics_table
)
from src.train import Trainer, set_seed, get_device, get_adaptive_batch_size, compute_val_metrics_wrapper
from src.encoder import count_parameters

# Paths
PROCESSED_DIR = os.path.join(PROJECT_ROOT, 'data', 'processed')
SPLITS_DIR = os.path.join(PROJECT_ROOT, 'data', 'splits')
CHECKPOINT_DIR = os.path.join(PROJECT_ROOT, 'checkpoints')
RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results')
FIGURES_DIR = os.path.join(RESULTS_DIR, 'figures')
TABLES_DIR = os.path.join(RESULTS_DIR, 'tables')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(TABLES_DIR, exist_ok=True)

# Configuration
SEED = 42
DEVICE = get_device('auto')
BATCH_SIZE = get_adaptive_batch_size(DEVICE, default=8)
print(f"Batch size: {BATCH_SIZE}")

## 4.1 Setup Datasets and Model

In [ ]:
set_seed(SEED)

# Load datasets
train_dataset = ACDCSegDataset(
    processed_dir=PROCESSED_DIR,
    split_file=os.path.join(SPLITS_DIR, 'train.json'),
    transform=get_train_transforms(),
)

val_dataset = ACDCSegDataset(
    processed_dir=PROCESSED_DIR,
    split_file=os.path.join(SPLITS_DIR, 'val.json'),
    transform=get_val_transforms(),
)

test_dataset = ACDCSegDataset(
    processed_dir=PROCESSED_DIR,
    split_file=os.path.join(SPLITS_DIR, 'test.json'),
    transform=get_val_transforms(),
)

print(f"Train: {len(train_dataset)} slices")
print(f"Val:   {len(val_dataset)} slices")
print(f"Test:  {len(test_dataset)} slices")

# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True)

In [ ]:
# Build model
model = SegmentationUNet(
    in_channels=1,
    num_classes=4,
    encoder_channels=[32, 64, 128, 256],
    dropout=0.1,
)

n_params = count_parameters(model)
print(f"Model parameters: {n_params:,} ({n_params/1e6:.2f}M)")

# Verify forward pass
dummy = torch.randn(2, 1, 256, 256)
with torch.no_grad():
    out = model(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}")
assert out.shape == (2, 4, 256, 256), f"Unexpected output shape: {out.shape}"
print("✓ Model forward pass verified")

## 4.2 Train Baseline (100% Labels)

In [ ]:
set_seed(SEED)

# Re-initialize model
model = SegmentationUNet(
    in_channels=1, num_classes=4,
    encoder_channels=[32, 64, 128, 256], dropout=0.1,
)

# Loss and optimizer
criterion = DiceCELoss(num_classes=4)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=200, eta_min=1e-6)

# Trainer
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    device=DEVICE,
    scheduler=scheduler,
    mixed_precision=True,
    checkpoint_dir=CHECKPOINT_DIR,
    log_dir=os.path.join(RESULTS_DIR, 'logs'),
    experiment_name='baseline_100pct',
)

# Train
history = trainer.train(
    train_loader=train_loader,
    val_loader=val_loader,
    n_epochs=200,
    early_stopping_patience=30,
    compute_metrics_fn=compute_val_metrics_wrapper,
    monitor_metric='Mean_Dice',
)

## 4.3 Evaluate Baseline on Test Set

In [ ]:
# Load best model
best_ckpt = torch.load(
    os.path.join(CHECKPOINT_DIR, 'baseline_100pct_best.pth'),
    map_location=DEVICE, weights_only=False
)
model.load_state_dict(best_ckpt['model_state_dict'])
model = model.to(DEVICE)
model.eval()

print(f"Best model from epoch {best_ckpt['epoch']}")

# Full test evaluation with HD95
all_preds = []
all_targets = []
all_patient_ids = []

with torch.no_grad():
    for batch in test_loader:
        images = batch['image'].to(DEVICE)
        
        if torch.cuda.is_available():
            with torch.cuda.amp.autocast():
                logits = model(images)
        else:
            logits = model(images)
        
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        targets = batch['mask'].numpy()
        
        all_preds.append(preds)
        all_targets.append(targets)
        all_patient_ids.extend(batch['patient_id'])

all_preds = np.concatenate(all_preds)
all_targets = np.concatenate(all_targets)

# Compute metrics
batch_metrics = compute_metrics_batch(
    all_preds, all_targets, all_patient_ids, compute_hd=True
)
patient_metrics = compute_patient_level_metrics(batch_metrics['per_sample'])

# Print results
print(format_metrics_table(patient_metrics, "Baseline U-Net (100% Labels) — Test Set"))

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(history['train_loss'], label='Train', color='steelblue')
axes[0].plot(history['val_loss'], label='Val', color='coral')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')
axes[0].legend()

axes[1].plot(history['val_dice'], color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Mean Dice')
axes[1].set_title('Validation Dice')

axes[2].plot(history['lr'], color='purple')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Learning Rate')
axes[2].set_title('Learning Rate Schedule')
axes[2].set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'baseline_training_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5.1 Limited-Label Experiments

In [ ]:
# Train baseline at each label fraction
label_fractions = [0.10, 0.25, 0.50]  # 100% already done above
all_results = {}

# Store 100% result
all_results[1.0] = {
    'mean': patient_metrics['mean'],
    'std': patient_metrics['std'],
    'n_patients_train': len(train_dataset.get_patient_ids()),
}

for frac in label_fractions:
    print(f"\n{'='*60}")
    print(f"Training baseline with {int(frac*100)}% labels")
    print(f"{'='*60}")
    
    set_seed(SEED)
    
    # Load limited dataset
    frac_name = f'train_{int(frac*100)}pct'
    frac_dataset = ACDCSegDataset(
        processed_dir=PROCESSED_DIR,
        split_file=os.path.join(SPLITS_DIR, f'{frac_name}.json'),
        transform=get_train_transforms(),
    )
    
    frac_loader = DataLoader(frac_dataset, batch_size=BATCH_SIZE, shuffle=True,
                             num_workers=4, pin_memory=True, drop_last=True)
    
    print(f"Training samples: {len(frac_dataset)} ({len(frac_dataset.get_patient_ids())} patients)")
    
    # Fresh model
    model_frac = SegmentationUNet(
        in_channels=1, num_classes=4,
        encoder_channels=[32, 64, 128, 256], dropout=0.1,
    )
    
    optimizer_frac = torch.optim.AdamW(model_frac.parameters(), lr=1e-4, weight_decay=1e-5)
    scheduler_frac = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_frac, T_max=200, eta_min=1e-6)
    
    trainer_frac = Trainer(
        model=model_frac,
        optimizer=optimizer_frac,
        criterion=DiceCELoss(num_classes=4),
        device=DEVICE,
        scheduler=scheduler_frac,
        mixed_precision=True,
        checkpoint_dir=CHECKPOINT_DIR,
        log_dir=os.path.join(RESULTS_DIR, 'logs'),
        experiment_name=f'baseline_{frac_name}',
    )
    
    history_frac = trainer_frac.train(
        train_loader=frac_loader,
        val_loader=val_loader,
        n_epochs=200,
        early_stopping_patience=30,
        compute_metrics_fn=compute_val_metrics_wrapper,
        monitor_metric='Mean_Dice',
    )
    
    # Evaluate on test
    best_ckpt_frac = torch.load(
        os.path.join(CHECKPOINT_DIR, f'baseline_{frac_name}_best.pth'),
        map_location=DEVICE, weights_only=False
    )
    model_frac.load_state_dict(best_ckpt_frac['model_state_dict'])
    model_frac = model_frac.to(DEVICE)
    model_frac.eval()
    
    preds_frac = []
    targets_frac = []
    pids_frac = []
    
    with torch.no_grad():
        for batch in test_loader:
            images = batch['image'].to(DEVICE)
            if torch.cuda.is_available():
                with torch.cuda.amp.autocast():
                    logits = model_frac(images)
            else:
                logits = model_frac(images)
            preds_frac.append(torch.argmax(logits, dim=1).cpu().numpy())
            targets_frac.append(batch['mask'].numpy())
            pids_frac.extend(batch['patient_id'])
    
    preds_frac = np.concatenate(preds_frac)
    targets_frac = np.concatenate(targets_frac)
    
    batch_m = compute_metrics_batch(preds_frac, targets_frac, pids_frac, compute_hd=True)
    patient_m = compute_patient_level_metrics(batch_m['per_sample'])
    
    print(format_metrics_table(patient_m, f"Baseline ({int(frac*100)}% Labels)"))
    
    all_results[frac] = {
        'mean': patient_m['mean'],
        'std': patient_m['std'],
        'n_patients_train': len(frac_dataset.get_patient_ids()),
    }

## 5.2 Label-Efficiency Curve

In [ ]:
# Plot label-efficiency curve
fractions = sorted(all_results.keys())
mean_dice = [all_results[f]['mean']['Mean_Dice'] for f in fractions]
std_dice = [all_results[f]['std']['Mean_Dice'] for f in fractions]
lv_dice = [all_results[f]['mean']['LV_Dice'] for f in fractions]
myo_dice = [all_results[f]['mean']['Myocardium_Dice'] for f in fractions]
rv_dice = [all_results[f]['mean']['RV_Dice'] for f in fractions]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Mean Dice curve
pct_labels = [int(f*100) for f in fractions]
axes[0].errorbar(pct_labels, mean_dice, yerr=std_dice, marker='o', linewidth=2,
                 capsize=5, color='steelblue', markersize=8)
axes[0].set_xlabel('% Training Labels', fontsize=12)
axes[0].set_ylabel('Mean Dice', fontsize=12)
axes[0].set_title('Baseline: Label-Efficiency Curve', fontsize=13)
axes[0].set_xticks(pct_labels)
axes[0].grid(True, alpha=0.3)
axes[0].set_ylim([0.4, 1.0])

# Per-class curves
axes[1].plot(pct_labels, lv_dice, 'o-', label='LV', color='#e74c3c', linewidth=2)
axes[1].plot(pct_labels, myo_dice, 's-', label='Myo', color='#2ecc71', linewidth=2)
axes[1].plot(pct_labels, rv_dice, '^-', label='RV', color='#3498db', linewidth=2)
axes[1].set_xlabel('% Training Labels', fontsize=12)
axes[1].set_ylabel('Dice Score', fontsize=12)
axes[1].set_title('Per-Class Dice vs Label Fraction', fontsize=13)
axes[1].set_xticks(pct_labels)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0.4, 1.0])

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'label_efficiency_curve_baseline.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save results table
table_rows = []
for frac in sorted(all_results.keys()):
    r = all_results[frac]
    row = {
        'Label_%': int(frac * 100),
        'N_patients': r['n_patients_train'],
        'Mean_Dice': f"{r['mean']['Mean_Dice']:.4f} ± {r['std']['Mean_Dice']:.4f}",
        'LV_Dice': f"{r['mean']['LV_Dice']:.4f} ± {r['std']['LV_Dice']:.4f}",
        'Myo_Dice': f"{r['mean']['Myocardium_Dice']:.4f} ± {r['std']['Myocardium_Dice']:.4f}",
        'RV_Dice': f"{r['mean']['RV_Dice']:.4f} ± {r['std']['RV_Dice']:.4f}",
    }
    # Add HD95 if available
    if 'Mean_HD95' in r['mean']:
        row['Mean_HD95'] = f"{r['mean']['Mean_HD95']:.2f} ± {r['std']['Mean_HD95']:.2f}"
    table_rows.append(row)

results_df = pd.DataFrame(table_rows)
print("\n" + "="*80)
print("TABLE B: Baseline Label-Efficiency Results")
print("="*80)
print(results_df.to_string(index=False))

# Save
results_df.to_csv(os.path.join(TABLES_DIR, 'baseline_label_efficiency.csv'), index=False)

with open(os.path.join(RESULTS_DIR, 'baseline_results.json'), 'w') as f:
    # Convert to serializable format
    serializable = {}
    for k, v in all_results.items():
        serializable[str(k)] = v
    json.dump(serializable, f, indent=2)

print("\n=== Step 4 (Baseline) + Step 5 (Limited Labels) COMPLETE ===")